In [ ]:
# --- Imports ---
import oci
from LoadProperties import LoadProperties
properties = LoadProperties()

from langchain.agents import initialize_agent, Tool, AgentType
from langchain.utilities import WikipediaAPIWrapper
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from IPython.display import Markdown, display

# --- Calculator Tool ---
def calculator_tool(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error in calculation: {e}"

# --- Wikipedia Tool ---
wiki = WikipediaAPIWrapper()

# --- Tools ---
tools = [
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Useful for solving basic math expressions like '12*4' or 'sqrt(64)'."
    ),
    Tool(
        name="Wikipedia",
        func=wiki.run,
        description="Useful for answering factual questions using Wikipedia content."
    )
]

# --- Define structured JSON output schema ---
response_schemas = [
    ResponseSchema(name="answer", description="The final answer to the user's question"),
    ResponseSchema(name="source", description="The tool(s) or source(s) used for this answer")
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

# --- OCI LLM ---
llm = ChatOCIGenAI(
    model_id='meta.llama-3.3-70b-instruct',
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',
    model_kwargs={"max_tokens": 600}
)

# --- Custom system prompt to enforce JSON format ---
system_prompt = f"""
You are a helpful reasoning agent. Use tools like Calculator and Wikipedia to answer the question.

At the end, always respond ONLY in the following JSON format:

{{
  "answer": "A complete answer to the question in plain language.",
  "source": "The name(s) of the tool(s) or source(s) used, like Calculator, Wikipedia, or both."
}}

Do not include any explanation or formatting outside this JSON object.
"""


# --- Initialize Agent ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
    agent_kwargs={"system_message": system_prompt}
)

# --- User Query ---
query = "What is 17 * 24 and who invented the telescope? Respond only with JSON keys 'answer' and 'source'."


# --- Run the agent ---
response = agent.invoke({"input": query})

parsed_output = output_parser.parse(response["output"])
print("\n✅ Final Parsed JSON Output:")
print(parsed_output)
display(Markdown(f"**Answer:** {parsed_output['answer']}  \n**Source:** {parsed_output['source']}"))

